### The Foundation – Pure Rotation.  
$$R_x(\alpha) = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\alpha & -\sin\alpha \\ 0 & \sin\alpha & \cos\alpha \end{bmatrix}$$ $$R_y(\beta) = \begin{bmatrix} \cos\beta & 0 & \sin\beta \\ 0 & 1 & 0 \\ -\sin\beta & 0 & \cos\beta \end{bmatrix}$$ $$R_z(\gamma) = \begin{bmatrix} \cos\gamma & -\sin\gamma & 0 \\ \sin\gamma & \cos\gamma & 0 \\ 0 & 0 & 1 \end{bmatrix}$$


### The Combined Rotation Matrix
To find the final orientation of a link in 3D space, we multiply the individual rotation matrices. Order matters! We will apply the rotations in the order of X, then Y, then Z. This is represented mathematically by multiplying from right to left: $R = R_z(\gamma) \cdot R_y(\beta) \cdot R_x(\alpha)$.

First, let's look at the expanded multiplication:

$$R_{zyx} = \begin{bmatrix} \cos\gamma & -\sin\gamma & 0 \\ \sin\gamma & \cos\gamma & 0 \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} \cos\beta & 0 & \sin\beta \\ 0 & 1 & 0 \\ -\sin\beta & 0 & \cos\beta \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos\alpha & -\sin\alpha \\ 0 & \sin\alpha & \cos\alpha \end{bmatrix}$$

Using the shorthand $c$ for cosine and $s$ for sine (e.g., $c\alpha = \cos\alpha$), the fully multiplied $3 \times 3$ rotation matrix becomes:

$$R_{zyx} = \begin{bmatrix} c\gamma c\beta & c\gamma s\beta s\alpha - s\gamma c\alpha & c\gamma s\beta c\alpha + s\gamma s\alpha \\ s\gamma c\beta & s\gamma s\beta s\alpha + c\gamma c\alpha & s\gamma s\beta c\alpha - c\gamma s\alpha \\ -s\beta & c\beta s\alpha & c\beta c\alpha \end{bmatrix}$$

In [16]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

# Define the rotation matrices
def rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

# Plotting function with interactive sliders
def plot_rotation(theta_x, theta_y, theta_z):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Original frame vectors (X=Red, Y=Green, Z=Blue)
    origin = np.array([0, 0, 0])
    x_axis = np.array([1, 0, 0])
    y_axis = np.array([0, 1, 0])
    z_axis = np.array([0, 0, 1])
    
    # Convert angles from degrees to radians
    tx, ty, tz = np.radians(theta_x), np.radians(theta_y), np.radians(theta_z)
    
    # Multiply the matrices (Order matters: Rz * Ry * Rx)
    R = rz(tz) @ ry(ty) @ rx(tx)
    
    # Apply rotation to our axes
    x_rot = R @ x_axis
    y_rot = R @ y_axis
    z_rot = R @ z_axis
    
    # Plot original axes (dashed)
    ax.quiver(*origin, *x_axis, color='r', linestyle='dashed', alpha=0.3)
    ax.quiver(*origin, *y_axis, color='g', linestyle='dashed', alpha=0.3)
    ax.quiver(*origin, *z_axis, color='b', linestyle='dashed', alpha=0.3)
    
    # Plot rotated axes (solid)
    ax.quiver(*origin, *x_rot, color='r', linewidth=3, label='X (Rotated)')
    ax.quiver(*origin, *y_rot, color='g', linewidth=3, label='Y (Rotated)')
    ax.quiver(*origin, *z_rot, color='b', linewidth=3, label='Z (Rotated)')
    
    # Formatting
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Rotation applied: Rx({theta_x}°) -> Ry({theta_y}°) -> Rz({theta_z}°)')
    ax.legend()
    plt.show()

# Setup UI sliders starting cleanly at 0 degrees
interact(plot_rotation, 
         theta_x=FloatSlider(value=0, min=-180, max=180, step=1, description='Rx (deg)'),
         theta_y=FloatSlider(value=0, min=-180, max=180, step=1, description='Ry (deg)'),
         theta_z=FloatSlider(value=0, min=-180, max=180, step=1, description='Rz (deg)'))

interactive(children=(FloatSlider(value=0.0, description='Rx (deg)', max=180.0, min=-180.0, step=1.0), FloatSl…

<function __main__.plot_rotation(theta_x, theta_y, theta_z)>

### Adding Translation (The $4 \times 4$ Matrix)
A $3 \times 3$ rotation matrix is great for pointing a joint in the right direction, but our robotic arm has physical length. We need to move our coordinate frame through space using a translation vector $P = [T_x, T_y, T_z]^T$.

To combine a $3 \times 3$ Rotation matrix ($R$) with a $3 \times 1$ Translation vector ($P$), we upgrade to a $4 \times 4$ **Homogeneous Transformation Matrix ($T$)**. We add a dummy bottom row `[0, 0, 0, 1]` simply to keep the matrix square, which allows us to multiply multiple joints together later!

$$T = \begin{bmatrix} R_{3\times3} & P_{3\times1} \\ 0_{1\times3} & 1 \end{bmatrix} = \begin{bmatrix} r_{11} & r_{12} & r_{13} & T_x \\ r_{21} & r_{22} & r_{23} & T_y \\ r_{31} & r_{32} & r_{33} & T_z \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

In [17]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Helper functions for rotation matrices
def rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

# Interactive plot combining Rotation and Translation
def plot_transformation(r_x, r_y, r_z, t_x, t_y, t_z):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Define the original frame (Origin and unit vectors)
    # Using 4D vectors [x, y, z, 1] for 4x4 matrix multiplication
    origin = np.array([0, 0, 0, 1])
    x_axis = np.array([1, 0, 0, 1])
    y_axis = np.array([0, 1, 0, 1])
    z_axis = np.array([0, 0, 1, 1])
    
    # 1. Calculate Rotation (3x3)
    rad_x, rad_y, rad_z = np.radians(r_x), np.radians(r_y), np.radians(r_z)
    R_3x3 = rz(rad_z) @ ry(rad_y) @ rx(rad_x)
    
    # 2. Build the 4x4 Homogeneous Transformation Matrix (T)
    T = np.eye(4)
    T[:3, :3] = R_3x3       # Insert 3x3 rotation into top left
    T[:3, 3] = [t_x, t_y, t_z] # Insert translation into top right
    
    # 3. Apply the Transformation Matrix to our frame
    new_origin = T @ origin
    new_x = T @ x_axis
    new_y = T @ y_axis
    new_z = T @ z_axis
    
    # Plot Original Base Frame (Faded, dashed)
    ax.quiver(0, 0, 0, 1, 0, 0, color='r', linestyle='dashed', alpha=0.2)
    ax.quiver(0, 0, 0, 0, 1, 0, color='g', linestyle='dashed', alpha=0.2)
    ax.quiver(0, 0, 0, 0, 0, 1, color='b', linestyle='dashed', alpha=0.2)
    
    # Plot New Transformed Frame (Solid)
    # Extract just the X,Y,Z components (indices 0,1,2) for plotting directions
    dx, dy, dz = (new_x - new_origin)[:3], (new_y - new_origin)[:3], (new_z - new_origin)[:3]
    ox, oy, oz = new_origin[:3]
    
    ax.quiver(ox, oy, oz, dx[0], dx[1], dx[2], color='r', linewidth=3, label='X Axis')
    ax.quiver(ox, oy, oz, dy[0], dy[1], dy[2], color='g', linewidth=3, label='Y Axis')
    ax.quiver(ox, oy, oz, dz[0], dz[1], dz[2], color='b', linewidth=3, label='Z Axis')
    
    # Draw a line from base origin to the new origin to show the translation "link"
    ax.plot([0, ox], [0, oy], [0, oz], color='black', linestyle='dotted', linewidth=2, label='Translation Vector')
    
    # Formatting
    limit = 5
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.set_zlim([-limit, limit])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Pose: T=({t_x},{t_y},{t_z}), R=({r_x}°,{r_y}°,{r_z}°)')
    ax.legend()
    plt.show()

# UI Setup - All starting at 0
interact(plot_transformation, 
         r_x=FloatSlider(value=0, min=-180, max=180, step=1, description='Rot X (deg)'),
         r_y=FloatSlider(value=0, min=-180, max=180, step=1, description='Rot Y (deg)'),
         r_z=FloatSlider(value=0, min=-180, max=180, step=1, description='Rot Z (deg)'),
         t_x=FloatSlider(value=0, min=-5, max=5, step=0.5, description='Trans X'),
         t_y=FloatSlider(value=0, min=-5, max=5, step=0.5, description='Trans Y'),
         t_z=FloatSlider(value=0, min=-5, max=5, step=0.5, description='Trans Z'))

interactive(children=(FloatSlider(value=0.0, description='Rot X (deg)', max=180.0, min=-180.0, step=1.0), Floa…

<function __main__.plot_transformation(r_x, r_y, r_z, t_x, t_y, t_z)>

### Calculating Forward Kinematics (FK)
Forward Kinematics is the process of finding the final position of the end-effector (the tool) by multiplying the transformation matrices of every joint from the base up. 

For our specific arm design—moving from the base, to a dual-motor shoulder, to the elbow, up to the single-motor wrist, and finally the tool—the equation looks like this:

$$T_{base}^{tool} = T_{base}^{shoulder} \cdot T_{shoulder}^{elbow} \cdot T_{elbow}^{wrist} \cdot T_{wrist}^{tool}$$

By extracting the top-right $3 \times 1$ vector from the final $T_{base}^{tool}$ matrix, we find the exact $X, Y, Z$ coordinates of our scooper in the real world.

### Decoding the Transformation Matrix
After multiplying the matrices of all joints, we receive the final homogeneous transformation matrix, $T$. It contains both the orientation (a $3 \times 3$ rotation matrix, $R$) and the position (a $3 \times 1$ translation vector, $P$) of our end effector.

$$T = \begin{bmatrix} r_{11} & r_{12} & r_{13} & p_x \\ r_{21} & r_{22} & r_{23} & p_y \\ r_{31} & r_{32} & r_{33} & p_z \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

#### 1. Deriving Position (XYZ)
Extracting the Cartesian coordinates in 3D space is straightforward. We pull the first three rows of the last column:
* **X Position:** $X = p_x$
* **Y Position:** $Y = p_y$
* **Z Position:** $Z = p_z$

#### 2. Deriving Orientation (Yaw, Pitch, Roll)
To find the orientation, we reverse-engineer the $3 \times 3$ rotation matrix. Assuming the standard $Z-Y-X$ Euler angle convention (Yaw-Pitch-Roll), we calculate the angles using the `atan2` function, which properly handles the signs of the quadrants to prevent ambiguity.

* **Pitch ($\beta$ - rotation around Y):** $$\beta = \text{atan2}\left(-r_{31}, \sqrt{r_{32}^2 + r_{33}^2}\right)$$

* **Roll ($\alpha$ - rotation around X):** $$\alpha = \text{atan2}\left(r_{32}, r_{33}\right)$$

* **Yaw ($\gamma$ - rotation around Z):** $$\gamma = \text{atan2}\left(r_{21}, r_{11}\right)$$

*Note: In the event of a singularity (Gimbal Lock) where $r_{31} = \pm 1$, the pitch is at $\pm 90^\circ$, and the roll and yaw axes align. A different mathematical check is required to resolve those specific edge cases.*

### Forward Kinematics of a 2-Link Planar Arm
Before moving to a full 6-DOF arm in 3D space, let's understand how Homogeneous Transformation Matrices chain together on a simple flat plane (2D). 

Our arm has two links ($L_1$ and $L_2$) and two joints ($\theta_1$ and $\theta_2$). 

#### 1. The Transformation Matrices
For each joint, we rotate around the Z-axis by $\theta$, and then translate along our new X-axis by the length of the link. The $4 \times 4$ matrices for Joint 1 and Joint 2 look like this:

**Joint 1 (Base to Elbow):**
$$T_0^1 = \begin{bmatrix} \cos\theta_1 & -\sin\theta_1 & 0 & L_1\cos\theta_1 \\ \sin\theta_1 & \cos\theta_1 & 0 & L_1\sin\theta_1 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

**Joint 2 (Elbow to Tool):**
$$T_1^2 = \begin{bmatrix} \cos\theta_2 & -\sin\theta_2 & 0 & L_2\cos\theta_2 \\ \sin\theta_2 & \cos\theta_2 & 0 & L_2\sin\theta_2 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

#### 2. Chaining the Matrices (The Forward Kinematics)
To find the final position of the tool relative to the base, we multiply the two matrices: $T_0^2 = T_0^1 \cdot T_1^2$.

When we multiply these out, the math simplifies using trigonometric identities (specifically the sum of angles formulas). The resulting transformation matrix is:

$$T_0^2 = \begin{bmatrix} \cos(\theta_1+\theta_2) & -\sin(\theta_1+\theta_2) & 0 & L_1\cos\theta_1 + L_2\cos(\theta_1+\theta_2) \\ \sin(\theta_1+\theta_2) & \cos(\theta_1+\theta_2) & 0 & L_1\sin\theta_1 + L_2\sin(\theta_1+\theta_2) \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

#### 3. Extracting the Final Coordinates
By looking at the last column of our final $T_0^2$ matrix, we get the exact equations for the $X$ and $Y$ coordinates of our tool!

* **X Position:** $X = L_1\cos\theta_1 + L_2\cos(\theta_1+\theta_2)$
* **Y Position:** $Y = L_1\sin\theta_1 + L_2\sin(\theta_1+\theta_2)$

Run the Python cell below to see these mathematical transformations visually calculate the end-effector position in real-time as you move the sliders.

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Link lengths
L1 = 1.0
L2 = 1.0

# Function to create a 4x4 transformation matrix for Z-axis rotation + X-axis translation
def get_transform(theta, length):
    # Convert degrees to radians
    rad = np.radians(theta)
    c, s = np.cos(rad), np.sin(rad)
    
    # Homogeneous Transformation Matrix
    return np.array([
        [c, -s, 0, length * c],
        [s,  c, 0, length * s],
        [0,  0, 1, 0],
        [0,  0, 0, 1]
    ])

# Function to calculate FK and plot the arm
def plot_2link_arm(theta1, theta2):
    # Base frame is at origin
    T0 = np.eye(4)
    origin = T0[:3, 3]
    
    # Transform from Base to Joint 1 (End of Link 1)
    T0_1 = get_transform(theta1, L1)
    joint1_pos = T0_1[:3, 3]
    
    # Transform from Joint 1 to Joint 2 (End of Link 2 / Tool)
    T1_2 = get_transform(theta2, L2)
    # Forward Kinematics: Multiply matrices to get Tool relative to Base
    T0_2 = T0_1 @ T1_2
    tool_pos = T0_2[:3, 3]
    
    # Extract X and Y for plotting
    x_coords = [origin[0], joint1_pos[0], tool_pos[0]]
    y_coords = [origin[1], joint1_pos[1], tool_pos[1]]
    
    # Plotting
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(x_coords, y_coords, '-ko', linewidth=4, markersize=10, mfc='red')
    
    # Highlight the links
    ax.plot([origin[0], joint1_pos[0]], [origin[1], joint1_pos[1]], color='blue', linewidth=4, label='Link 1')
    ax.plot([joint1_pos[0], tool_pos[0]], [joint1_pos[1], tool_pos[1]], color='green', linewidth=4, label='Link 2')
    
    # Formatting
    ax.set_xlim([-2.5, 2.5])
    ax.set_ylim([-2.5, 2.5])
    ax.grid(True)
    ax.set_title(f'2-Link FK: Tool at X={tool_pos[0]:.2f}, Y={tool_pos[1]:.2f}')
    ax.legend()
    plt.show()

# Setup interactive sliders
interact(plot_2link_arm, 
         theta1=FloatSlider(value=0, min=-180, max=180, step=1, description='Joint 1 (deg)'),
         theta2=FloatSlider(value=0, min=-180, max=180, step=1, description='Joint 2 (deg)'))

interactive(children=(FloatSlider(value=0.0, description='Joint 1 (deg)', max=180.0, min=-180.0, step=1.0), Fl…

<function __main__.plot_2link_arm(theta1, theta2)>

### The Classical (Standard) DH Convention
In the Classical Denavit-Hartenberg convention (popularized by Spong and Siciliano), frame $\{i\}$ is attached rigidly to link $i$ at the distal joint. 

The transformation from frame $\{i-1\}$ to frame $\{i\}$ is achieved through a different sequence of four operations compared to the modified version:
1. $R_z(\theta_i)$: Rotate about $Z_{i-1}$ by $\theta_i$ (Joint angle)
2. $T_z(d_i)$: Translate along $Z_{i-1}$ by $d_i$ (Link offset)
3. $T_x(a_i)$: Translate along $X_i$ by $a_i$ (Link length)
4. $R_x(\alpha_i)$: Rotate about $X_i$ by $\alpha_i$ (Link twist)

Multiplying these matrices together ($R_z \cdot T_z \cdot T_x \cdot R_x$) gives us the Classical DH Transformation Matrix:

$$T_{i-1}^{i} = \begin{bmatrix} \cos\theta_i & -\sin\theta_i \cos\alpha_i & \sin\theta_i \sin\alpha_i & a_i \cos\theta_i \\ \sin\theta_i & \cos\theta_i \cos\alpha_i & -\cos\theta_i \sin\alpha_i & a_i \sin\theta_i \\ 0 & \sin\alpha_i & \cos\alpha_i & d_i \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

### 6-DOF Kinematic Parameters (Classical DH)
Here is the Standard DH table for our 6-DOF manipulator, assuming a standard spherical wrist design.

* **$\theta_i$ (Joint Angle):** Rotation around $Z_{i-1}$
* **$d_i$ (Link Offset):** Translation along $Z_{i-1}$
* **$a_i$ (Link Length):** Translation along $X_i$
* **$\alpha_i$ (Link Twist):** Rotation around $X_i$

| $i$ | Joint | $\theta_i$ (deg) | $d_i$ | $a_i$ | $\alpha_i$ (deg) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | Base (Yaw) | $\theta_1$ | $L_1$ | 0 | 90 |
| 2 | Shoulder (Pitch) | $\theta_2$ | 0 | $L_2$ | 0 |
| 3 | Elbow (Pitch) | $\theta_3$ | 0 | 0 | 90 |
| 4 | Wrist 1 (Roll) | $\theta_4$ | $L_4$ | 0 | -90 |
| 5 | Wrist 2 (Pitch) | $\theta_5$ | 0 | 0 | 90 |
| 6 | Wrist 3 (Roll) | $\theta_6$ | $L_6$ | 0 | 0 |


### Visualizing Tool Orientation (Yaw, Pitch, Roll)
Our final Homogeneous Transformation Matrix ($T_6$) tells us exactly where the end-effector is and how it is rotated. 

While the last column gives us the $X, Y, Z$ position, the first three columns actually contain the unit vectors that define the tool's orientation in 3D space!

$$T_6 = \begin{bmatrix} X_{x} & Y_{x} & Z_{x} & P_x \\ X_{y} & Y_{y} & Z_{y} & P_y \\ X_{z} & Y_{z} & Z_{z} & P_z \\ 0 & 0 & 0 & 1 \end{bmatrix}$$

By extracting these first three columns, we can plot a visual coordinate frame right at the tip of the robot. 
* **Red Arrow (X-axis):** The Normal vector.
* **Green Arrow (Y-axis):** The Sliding vector.
* **Blue Arrow (Z-axis):** The Approach vector (where the tool is pointing).

Watching how these three arrows spin as we move the joints gives us a perfect visualization of our Yaw, Pitch, and Roll!

In [19]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Define physical link lengths
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# Classical DH Transformation Matrix Function
def classical_dh_matrix(theta_deg, d, a, alpha_deg):
    theta = np.radians(theta_deg)
    alpha = np.radians(alpha_deg)
    
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [0,   sa,       ca,      d],
        [0,   0,        0,       1]
    ])

# Forward Kinematics and 3D Plotting
def plot_classical_6dof_with_ypr(t1, t2, t3, t4, t5, t6):
    # Transformation Matrices
    T0_1 = classical_dh_matrix(t1, L1,  0,   90)
    T1_2 = classical_dh_matrix(t2,  0, L2,    0)
    T2_3 = classical_dh_matrix(t3,  0,  0,   90)
    T3_4 = classical_dh_matrix(t4, L4,  0,  -90)
    T4_5 = classical_dh_matrix(t5,  0,  0,   90)
    T5_6 = classical_dh_matrix(t6, L6,  0,    0)

    # Chaining Forward Kinematics
    T0 = np.eye(4)
    T1 = T0 @ T0_1
    T2 = T1 @ T1_2
    T3 = T2 @ T2_3
    T4 = T3 @ T3_4
    T5 = T4 @ T4_5
    T6 = T5 @ T5_6 # Final End Effector Pose

    # Extract positions for plotting links
    joints = np.array([
        T0[:3, 3], T1[:3, 3], T2[:3, 3], 
        T3[:3, 3], T4[:3, 3], T5[:3, 3], T6[:3, 3]
    ])
    
    x, y, z = joints[:, 0], joints[:, 1], joints[:, 2]

    # Plot Setup
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot robot links and joints
    ax.plot(x, y, z, '-o', color='black', linewidth=5, markersize=8, mfc='orange')
    
    # Highlight End Effector Origin
    tool_pos = T6[:3, 3]
    ax.scatter(*tool_pos, color='black', s=50)

    # TOOL ORIENTATION VISUALIZATION (YPR)
    arrow_len = 1.5
    
    # Extract the orientation vectors from the first 3 columns of T6
    tool_x = T6[:3, 0] * arrow_len
    tool_y = T6[:3, 1] * arrow_len
    tool_z = T6[:3, 2] * arrow_len
    
    # Plot the coordinate frame arrows at the tool position
    ax.quiver(*tool_pos, *tool_x, color='red', linewidth=3, label='Tool X')
    ax.quiver(*tool_pos, *tool_y, color='green', linewidth=3, label='Tool Y')
    ax.quiver(*tool_pos, *tool_z, color='blue', linewidth=3, label='Tool Z (Approach)')

    # Formatting limits for consistent view
    ax.set_xlim([-6, 6])
    ax.set_ylim([-6, 6])
    ax.set_zlim([ 0, 8])
    ax.set_xlabel('X Axis')
    ax.set_ylabel('Y Axis')
    ax.set_zlabel('Z Axis')
    
    ax.set_title(f'Tool Pos: X={tool_pos[0]:.1f}, Y={tool_pos[1]:.1f}, Z={tool_pos[2]:.1f}')
    ax.legend(loc='upper left')
    plt.show()

# Set up interactive sliders
interact(plot_classical_6dof_with_ypr,
         t1=FloatSlider(value=0, min=-180, max=180, step=1, description='Base (Yaw)'),
         t2=FloatSlider(value=0, min=-180, max=180, step=1, description='Shoulder(Pitch)'),
         t3=FloatSlider(value=0, min=-180, max=180, step=1, description='Elbow (Pitch)'),
         t4=FloatSlider(value=0, min=-180, max=180, step=1, description='Wrist 1 (Roll)'),
         t5=FloatSlider(value=0, min=-90,  max=90,  step=1, description='Wrist 2(Pitch)'),
         t6=FloatSlider(value=0, min=-180, max=180, step=1, description='Wrist 3 (Roll)'))

interactive(children=(FloatSlider(value=0.0, description='Base (Yaw)', max=180.0, min=-180.0, step=1.0), Float…

<function __main__.plot_classical_6dof_with_ypr(t1, t2, t3, t4, t5, t6)>

### Inverse Kinematics (IK): The Analytical Approach For 2 Link
In Forward Kinematics, we knew the angles ($\theta$) and wanted to find the position ($X, Y$). In Inverse Kinematics, we are given a target position ($X, Y$) and must solve for the joint angles ($\theta_1, \theta_2$) required to reach it.

For a simple 2-link planar arm, we can solve this exactly using trigonometry—specifically, the **Law of Cosines**.

Imagine a triangle formed by our two links ($L_1, L_2$) and a straight line drawn from the base origin to our target coordinate ($X, Y$). The length of that line to the target is the hypotenuse, $r$.

$$r = \sqrt{X^2 + Y^2}$$

#### Step 1: Solving for the Elbow ($\theta_2$)
Using the Law of Cosines on our triangle, we can find the angle of the elbow joint. 
$$r^2 = L_1^2 + L_2^2 - 2 L_1 L_2 \cos(180^\circ - \theta_2)$$

Using the cosine identity $\cos(180^\circ - x) = -\cos(x)$, this simplifies beautifully to:
$$X^2 + Y^2 = L_1^2 + L_2^2 + 2 L_1 L_2 \cos\theta_2$$

We isolate $\cos\theta_2$ (let's call it $c_2$):
$$c_2 = \frac{X^2 + Y^2 - L_1^2 - L_2^2}{2 L_1 L_2}$$

To find the actual angle, we use the `atan2` function, which requires both the sine and cosine. We find sine ($s_2$) using the Pythagorean identity $\sin^2 + \cos^2 = 1$:
$$s_2 = \pm\sqrt{1 - c_2^2}$$
*(Note the $\pm$. This proves there are two valid solutions for reaching a point: "Elbow Up" and "Elbow Down"!)*

$$\theta_2 = \text{atan2}(s_2, c_2)$$

#### Step 2: Solving for the Shoulder ($\theta_1$)
Once we have $\theta_2$, finding $\theta_1$ is a matter of subtracting the inner angle of our triangle from the total angle of the target relative to the base.

$$\theta_1 = \text{atan2}(Y, X) - \text{atan2}(L_2 \sin\theta_2, L_1 + L_2 \cos\theta_2)$$


[Visit For interactive visualization webapp](https://arm-kinematics-webapp.vercel.app/#/ik-2d)

In [20]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Link lengths
L1 = 2.0
L2 = 2.0

def plot_analytical_ik(target_x, target_y):
    # 1. Check if the target is reachable
    distance_to_target = np.sqrt(target_x**2 + target_y**2)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    
    if distance_to_target > (L1 + L2):
        # Target is too far away
        ax.scatter(target_x, target_y, color='red', s=100, marker='x')
        ax.text(0, 0, "UNREACHABLE TARGET", color='red', fontsize=14, ha='center')
        x_coords, y_coords = [0, 0, 0], [0, 0, 0] # Collapse arm
        title = "Error: Out of Workspace"
        
    else:
        # 2. Calculate Theta 2 (Elbow)
        # c2 = (X^2 + Y^2 - L1^2 - L2^2) / (2 * L1 * L2)
        c2 = (target_x**2 + target_y**2 - L1**2 - L2**2) / (2 * L1 * L2)
        
        # We will use the positive root for "Elbow Down" configuration
        s2 = np.sqrt(1 - c2**2) 
        theta2 = np.arctan2(s2, c2)
        
        # 3. Calculate Theta 1 (Shoulder)
        theta1 = np.arctan2(target_y, target_x) - np.arctan2(L2 * s2, L1 + L2 * c2)
        
        # 4. Use FK to find the exact joint positions for plotting
        joint1_x = L1 * np.cos(theta1)
        joint1_y = L1 * np.sin(theta1)
        
        tool_x = joint1_x + L2 * np.cos(theta1 + theta2)
        tool_y = joint1_y + L2 * np.sin(theta1 + theta2)
        
        x_coords = [0, joint1_x, tool_x]
        y_coords = [0, joint1_y, tool_y]
        
        # Convert to degrees for the title
        t1_deg = np.degrees(theta1)
        t2_deg = np.degrees(theta2)
        title = f'IK Solved: θ1 = {t1_deg:.1f}°, θ2 = {t2_deg:.1f}°'
        
        # Plot target goal as a green star
        ax.scatter(target_x, target_y, color='green', s=200, marker='*', zorder=5, label='Target Goal')

    # Plotting the arm
    ax.plot(x_coords, y_coords, '-ko', linewidth=4, markersize=10, mfc='red', zorder=4)
    ax.plot([0, x_coords[1]], [0, y_coords[1]], color='blue', linewidth=4, label='Link 1')
    ax.plot([x_coords[1], x_coords[2]], [y_coords[1], y_coords[2]], color='orange', linewidth=4, label='Link 2')
    
    # Formatting
    limit = 5
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.grid(True)
    ax.set_title(title)
    ax.legend(loc='upper right')
    plt.show()

# Setup sliders for Target X and Y
interact(plot_analytical_ik, 
         target_x=FloatSlider(value=2.0, min=-5, max=5, step=0.1, description='Target X'),
         target_y=FloatSlider(value=2.0, min=-5, max=5, step=0.1, description='Target Y'))

interactive(children=(FloatSlider(value=2.0, description='Target X', max=5.0, min=-5.0), FloatSlider(value=2.0…

<function __main__.plot_analytical_ik(target_x, target_y)>

### Analytical 6-DOF IK: Kinematic Decoupling
Solving six non-linear trigonometry equations simultaneously is nearly impossible. Thankfully, most 6-DOF industrial arms are designed with a **Spherical Wrist** (where axes 4, 5, and 6 intersect at a single point). 

Because of this physical design, we can use **Pieper's Theorem** to "decouple" the robot into two halves:
1. **Position:** Joints 1, 2, and 3 control the $X, Y, Z$ position of the wrist center.
2. **Orientation:** Joints 4, 5, and 6 control the Roll, Pitch, and Yaw of the tool.

#### Step 1: Finding the Wrist Center ($P_{wc}$)
If we know our target tool position ($P_{tool}$) and the direction the tool needs to point (our Approach Vector, $R_{z}$), we can "step back" along the tool to find exactly where the wrist must be located.

$$P_{wc} = P_{tool} - d_6 \cdot R_{z}$$
*(Where $d_6$ is the physical length of the final tool link).*

By finding $P_{wc}$, we have completely removed the last three joints from the equation! We now only need to calculate $\theta_1, \theta_2,$ and $\theta_3$ to reach the Wrist Center.  


#### Step 2: Solving for $\theta_1, \theta_2, \theta_3$
Now that we have the target $X, Y, Z$ coordinates for our Wrist Center ($P_{wc}$), solving the first three joints is remarkably similar to our 2-link planar arm.

**1. Base Rotation ($\theta_1$):**
We simply look at the $X$ and $Y$ coordinates of the wrist center from a top-down view.
$$\theta_1 = \text{atan2}(P_{wc,y}, P_{wc,x})$$

**2. The 2-Link Triangle ($\theta_2$ and $\theta_3$):**
Once the base rotates to face the target, Joints 2 and 3 operate on a flat 2D plane! 
* The horizontal distance to the target is $r = \sqrt{P_{wc,x}^2 + P_{wc,y}^2}$
* The vertical distance from the shoulder is $s = P_{wc,z} - L_1$

We now have a triangle with a hypotenuse reaching to the wrist center. We can use the exact same **Law of Cosines** we derived earlier to solve for the elbow ($\theta_3$) and shoulder ($\theta_2$) angles to reach this point.

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Physical link lengths
L1, L2, L4 = 2.0, 3.0, 2.5 
# (Note: L4 acts as our "forearm" reaching to the wrist center)

def plot_decoupled_ik(target_x, target_y, target_z):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 1. Base Rotation (Theta 1)
    # Simply face the target in the X/Y plane
    theta1 = np.arctan2(target_y, target_x)
    
    # 2. Convert to 2D Planar Problem (r, z)
    # r is the horizontal distance from the base origin
    r = np.sqrt(target_x**2 + target_y**2)
    # s is the vertical distance from the shoulder joint
    s = target_z - L1
    
    # Check if target is reachable by checking the hypotenuse
    hypotenuse = np.sqrt(r**2 + s**2)
    
    if hypotenuse > (L2 + L4) or hypotenuse < abs(L2 - L4):
        ax.text(0, 0, target_z, "UNREACHABLE", color='red', fontsize=16)
        x_coords, y_coords, z_coords = [0], [0], [0]
        title = "Error: Wrist Center out of bounds"
    else:
        # 3. Law of Cosines for the Elbow (Theta 3)
        c3 = (r**2 + s**2 - L2**2 - L4**2) / (2 * L2 * L4)
        s3 = -np.sqrt(1 - c3**2) # Using negative root for Elbow Up configuration
        theta3_geom = np.arctan2(s3, c3)
        
        # 4. Solve for Shoulder (Theta 2)
        theta2_geom = np.arctan2(s, r) - np.arctan2(L4 * s3, L2 + L4 * c3)
        
        # --- Map Geometric Angles to 3D Space ---
        # Shoulder position
        shoulder_x = 0
        shoulder_y = 0
        shoulder_z = L1
        
        # Elbow Position
        elbow_r = L2 * np.cos(theta2_geom)
        elbow_z = L1 + L2 * np.sin(theta2_geom)
        elbow_x = elbow_r * np.cos(theta1)
        elbow_y = elbow_r * np.sin(theta1)
        
        # Wrist Center Position (Should perfectly match Target)
        wc_r = elbow_r + L4 * np.cos(theta2_geom + theta3_geom)
        wc_z = elbow_z + L4 * np.sin(theta2_geom + theta3_geom)
        wc_x = wc_r * np.cos(theta1)
        wc_y = wc_r * np.sin(theta1)
        
        x_coords = [0, shoulder_x, elbow_x, wc_x]
        y_coords = [0, shoulder_y, elbow_y, wc_y]
        z_coords = [0, shoulder_z, elbow_z, wc_z]
        
        t1_deg, t2_deg, t3_deg = np.degrees(theta1), np.degrees(theta2_geom), np.degrees(theta3_geom)
        title = f'Analytical IK | θ1={t1_deg:.1f}°, θ2={t2_deg:.1f}°, θ3={t3_deg:.1f}°'
        
        # Plot target
        ax.scatter(target_x, target_y, target_z, color='green', s=200, marker='*', label='Target Wrist Center')

    # Plot Arm Links
    ax.plot(x_coords, y_coords, z_coords, '-ko', linewidth=5, markersize=8, mfc='orange')
    
    # Formatting
    limit = 6
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.set_zlim([0, 8])
    ax.set_xlabel('X Axis')
    ax.set_ylabel('Y Axis')
    ax.set_zlabel('Z Axis')
    ax.set_title(title)
    ax.legend()
    plt.show()

# Setup interactive sliders for the Wrist Center target
interact(plot_decoupled_ik,
         target_x=FloatSlider(value=3.0, min=-6, max=6, step=0.1, description='Wrist X'),
         target_y=FloatSlider(value=2.0, min=-6, max=6, step=0.1, description='Wrist Y'),
         target_z=FloatSlider(value=4.0, min=0, max=8, step=0.1, description='Wrist Z'))

interactive(children=(FloatSlider(value=3.0, description='Wrist X', max=6.0, min=-6.0), FloatSlider(value=2.0,…

<function __main__.plot_decoupled_ik(target_x, target_y, target_z)>

### Numerical IK using SciPy
While we can manually calculate the Jacobian matrix to step toward our target, Python's `scipy.optimize` library has extremely fast, mathematically robust solvers built right in. 

We simply give SciPy an **Error Function** (the distance between where the robot is currently pointing and where we *want* it to point). SciPy will automatically test joint angles, calculate the partial derivatives, and find the exact $\theta$ values that drop our error to zero.

In [22]:
import sympy as sp
from IPython.display import display, Math

# 1. Define our symbolic variables
t1, t2, t3, t4, t5, t6 = sp.symbols('theta_1 theta_2 theta_3 theta_4 theta_5 theta_6')
L1, L2, L4, L6 = sp.symbols('L_1 L_2 L_4 L_6')

# 2. Classical DH Matrix Generator (Symbolic)
def sym_dh(theta, d, a, alpha_deg):
    # Convert degrees to radians exactly
    alpha = alpha_deg * sp.pi / 180 
    
    return sp.Matrix([
        [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha),  sp.sin(theta)*sp.sin(alpha), a*sp.cos(theta)],
        [sp.sin(theta),  sp.cos(theta)*sp.cos(alpha), -sp.cos(theta)*sp.sin(alpha), a*sp.sin(theta)],
        [0,              sp.sin(alpha),                sp.cos(alpha),               d],
        [0,              0,                            0,                           1]
    ])

# 3. Create the individual joint matrices
T0_1 = sym_dh(t1, L1,  0,  90)
T1_2 = sym_dh(t2,  0, L2,   0)
T2_3 = sym_dh(t3,  0,  0,  90)
T3_4 = sym_dh(t4, L4,  0, -90)
T4_5 = sym_dh(t5,  0,  0,  90)
T5_6 = sym_dh(t6, L6,  0,   0)

# 4. Show Kinematic Decoupling: Position (Base to Wrist Center)
T0_3 = sp.simplify(T0_1 * T1_2 * T2_3)

print("--- Step 1: Equations for the Wrist Center (First 3 Joints) ---")
# Extract the X, Y, Z position equations from the last column
wc_x = T0_3[0, 3]
wc_y = T0_3[1, 3]
wc_z = T0_3[2, 3]

display(Math(f'X_{{wc}} = {sp.latex(wc_x)}'))
display(Math(f'Y_{{wc}} = {sp.latex(wc_y)}'))
display(Math(f'Z_{{wc}} = {sp.latex(wc_z)}'))

# 5. Show Orientation (Wrist to Tool)
T3_6 = sp.simplify(T3_4 * T4_5 * T5_6)
print("\n--- Step 2: The Spherical Wrist Orientation Matrix ---")
display(Math(sp.latex(T3_6)))

--- Step 1: Equations for the Wrist Center (First 3 Joints) ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- Step 2: The Spherical Wrist Orientation Matrix ---


<IPython.core.display.Math object>

In [23]:
import sympy as sp
from IPython.display import display, Math

# 1. Define our symbolic variables
t1, t2, t3, t4, t5, t6 = sp.symbols('theta_1 theta_2 theta_3 theta_4 theta_5 theta_6')

# 2. Insert real link numbers to shrink the equations
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# 3. Classical DH Matrix Generator
def sym_dh(theta, d, a, alpha_deg):
    alpha = alpha_deg * sp.pi / 180 
    return sp.Matrix([
        [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha),  sp.sin(theta)*sp.sin(alpha), a*sp.cos(theta)],
        [sp.sin(theta),  sp.cos(theta)*sp.cos(alpha), -sp.cos(theta)*sp.sin(alpha), a*sp.sin(theta)],
        [0,              sp.sin(alpha),                sp.cos(alpha),               d],
        [0,              0,                            0,                           1]
    ])

# 4. Create the individual joint matrices
T0_1 = sym_dh(t1, L1,  0,  90)
T1_2 = sym_dh(t2,  0, L2,   0)
T2_3 = sym_dh(t3,  0,  0,  90)
T3_4 = sym_dh(t4, L4,  0, -90)
T4_5 = sym_dh(t5,  0,  0,  90)
T5_6 = sym_dh(t6, L6,  0,   0)

# Multiply everything to get the full 6-DOF Forward Kinematics
T0_6 = T0_1 * T1_2 * T2_3 * T3_4 * T4_5 * T5_6

# ==========================================
# GOAL 1: THE HOME POSITION (All Thetas = 0)
# ==========================================
print("--- Step 1: The 'Home Position' Matrix (All Thetas = 0) ---")
T_home = T0_6.subs({t1: 0, t2: 0, t3: 0, t4: 0, t5: 0, t6: 0})
display(Math(sp.latex(T_home)))


# ==========================================
# GOAL 2: THE SHORTHAND DERIVATION MATRIX
# ==========================================
print("\n--- Step 2: Extracting the Derivation Equations (Using c and s shorthand) ---")

# We first let SymPy simplify the trigonometry (combining angles)
T0_6_simplified = sp.simplify(T0_6)

# Create a substitution dictionary for the shorthand
subs_dict = {}
# Add the combined angle shorthand for Joints 2 and 3 (c_23, s_23)
subs_dict[sp.cos(t2 + t3)] = sp.Symbol('c_{23}')
subs_dict[sp.sin(t2 + t3)] = sp.Symbol('s_{23}')

# Add standard shorthand for all individual joints
for i, t in enumerate([t1, t2, t3, t4, t5, t6], start=1):
    subs_dict[sp.cos(t)] = sp.Symbol(f'c_{i}')
    subs_dict[sp.sin(t)] = sp.Symbol(f's_{i}')

# Apply the shorthand to our simplified matrix
T0_6_shorthand = T0_6_simplified.subs(subs_dict)

# Display just the final X, Y, Z equations from the 4th column
display(Math(f'X = {sp.latex(T0_6_shorthand[0, 3])}'))
display(Math(f'Y = {sp.latex(T0_6_shorthand[1, 3])}'))
display(Math(f'Z = {sp.latex(T0_6_shorthand[2, 3])}'))

--- Step 1: The 'Home Position' Matrix (All Thetas = 0) ---


<IPython.core.display.Math object>


--- Step 2: Extracting the Derivation Equations (Using c and s shorthand) ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Deriving the Inverse Kinematics Algebra
If we look at the raw equations generated by our Forward Kinematics matrix, solving for the first three angles seems impossible because $\theta_4$ and $\theta_5$ are tangled up in the $X$ and $Y$ coordinates.

**The Decoupling Trick:** Our final link ($L_6$) is $1.0$ unit long. Every term in our equations multiplied by $1.0$ represents the tool pointing away from the wrist. If we subtract the tool's offset vector from our target, we are left with the coordinates for the **Wrist Center ($X_{wc}, Y_{wc}, Z_{wc}$)**.

Let's look at the remaining terms that define the wrist center (using our physical links $L_2 = 3.0$ and $L_4 = 2.5$):

* $X_{wc} = 3.0c_1c_2 + 2.5c_1s_{23}$
* $Y_{wc} = 3.0s_1c_2 + 2.5s_1s_{23}$
* $Z_{wc} = -2.5c_{23} + 3.0s_2 + 2.0$

Now, we can derive the exact algebraic equations for our first three motors!

---

#### 1. Deriving Base Yaw ($\theta_1$)
To find the base angle, we divide $Y_{wc}$ by $X_{wc}$. Notice how the complex terms beautifully cancel each other out!

$$\frac{Y_{wc}}{X_{wc}} = \frac{s_1(3.0c_2 + 2.5s_{23})}{c_1(3.0c_2 + 2.5s_{23})} = \frac{s_1}{c_1} = \tan(\theta_1)$$

Using the arctangent function, we get our first exact equation:
**$$\theta_1 = \text{atan2}(Y_{wc}, X_{wc})$$**

---

#### 2. Deriving Elbow Pitch ($\theta_3$)
Now we project our target onto a 2D plane. Let $r$ be the horizontal distance from the base, and $s$ be the vertical distance from the shoulder pivot (at $Z = 2.0$).

* $r = \sqrt{X_{wc}^2 + Y_{wc}^2} = 3.0c_2 + 2.5s_{23}$
* $s = Z_{wc} - 2.0 = 3.0s_2 - 2.5c_{23}$

If we square both and add them together ($r^2 + s^2$), we are calculating the square of the hypotenuse reaching to the wrist. Let's expand the algebra:

$$r^2 + s^2 = (3.0c_2 + 2.5s_{23})^2 + (3.0s_2 - 2.5c_{23})^2$$
$$= 9.0(c_2^2 + s_2^2) + 6.25(s_{23}^2 + c_{23}^2) + 15.0(c_2s_{23} - s_2c_{23})$$

Using standard trigonometric identities ($\cos^2 + \sin^2 = 1$), the first two chunks collapse into pure link lengths. The final chunk uses the angle subtraction identity ($\cos(A)\sin(B) - \sin(A)\cos(B) = \sin(B-A)$), which gives us:

$$r^2 + s^2 = 9.0 + 6.25 + 15.0\sin(\theta_3)$$

*(Note: Unlike the 2-link planar arm which used $\cos(\theta_3)$, our 3D arm uses $\sin(\theta_3)$ due to the $90^\circ$ twists in our classical DH architecture!)*

We can now isolate $\theta_3$:
**$$\sin(\theta_3) = \frac{r^2 + s^2 - 15.25}{15.0}$$**
$$\cos(\theta_3) = \pm\sqrt{1 - \sin^2(\theta_3)}$$
**$$\theta_3 = \text{atan2}(\sin(\theta_3), \cos(\theta_3))$$**

---

#### 3. Deriving Shoulder Pitch ($\theta_2$)
With $\theta_3$ solved, we can plug it back into our $r$ and $s$ equations to find $\theta_2$. Let's expand $s_{23}$ and $c_{23}$ using angle addition formulas:

* $r = c_2(3.0 + 2.5s_3) + s_2(2.5c_3)$
* $s = s_2(3.0 + 2.5s_3) - c_2(2.5c_3)$

To simplify, let's define two new constants based on things we already know:
$A = 3.0 + 2.5s_3$
$B = 2.5c_3$

Our system becomes:
* $r = A c_2 + B s_2$
* $s = A s_2 - B c_2$

By solving this system of equations for $c_2$ and $s_2$, we get:
$c_2 = \frac{A r - B s}{A^2 + B^2}$
$s_2 = \frac{B r + A s}{A^2 + B^2}$

And finally, our exact equation for the shoulder:
**$$\theta_2 = \text{atan2}(B r + A s, A r - B s)$$**

### Solving for Orientation (The Spherical Wrist)
We have successfully calculated $\theta_1, \theta_2,$ and $\theta_3$ to place our wrist center exactly where it needs to be. Now, we must bend the wrist ($\theta_4, \theta_5, \theta_6$) to match the target's Roll, Pitch, and Yaw.

Our total desired rotation is $R_{target}$. We know that the total rotation is just the base rotation multiplied by the wrist rotation:
$$R_{target} = R_{base}^{elbow} \cdot R_{wrist}^{tool}$$

Since we already know the first three joint angles, we can calculate $R_{base}^{elbow}$ ($R_0^3$). To isolate the wrist, we simply multiply both sides by the inverse (or transpose) of $R_0^3$:
$$R_{wrist}^{tool} = (R_0^3)^T \cdot R_{target}$$

#### Extracting $\theta_4, \theta_5, \theta_6$
By multiplying out the DH matrices for joints 4, 5, and 6, we get a $3 \times 3$ symbolic matrix that looks like a standard Euler Z-Y-Z sequence. Let's call our calculated numerical wrist matrix $R$:

$$R_{3}^{6} = \begin{bmatrix} r_{11} & r_{12} & c_4 s_5 \\ r_{21} & r_{22} & s_4 s_5 \\ -s_5 c_6 & s_5 s_6 & c_5 \end{bmatrix}$$

From this, extracting the final three angles is beautiful and exact:
1. **Wrist 2 Pitch ($\theta_5$):** We look at the bottom right corner.
   $$\theta_5 = \text{atan2}(\sqrt{r_{13}^2 + r_{23}^2}, r_{33})$$
2. **Wrist 1 Roll ($\theta_4$):** We use the top two elements of the last column.
   $$\theta_4 = \text{atan2}(r_{23}, r_{13})$$
3. **Wrist 3 Roll ($\theta_6$):** We use the first two elements of the bottom row.
   $$\theta_6 = \text{atan2}(r_{32}, -r_{31})$$

*(Note: If $\theta_5 = 0$, the wrist is straight, and axes 4 and 6 align perfectly. This is a singularity called **Gimbal Lock**. In this state, the math breaks down because there are infinite ways to spin joints 4 and 6 to get the same result. Our code handles this by locking $\theta_4$ to zero and doing all the spinning with $\theta_6$.)*

[Visit for interactive webapp](https://arm-kinematics-webapp.vercel.app/#/ik-6dof)

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Physical Link Lengths
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# --- Helper Functions ---
def rot_x(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def rot_y(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rot_z(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def dh_matrix(theta_rad, d, a, alpha_deg):
    alpha = np.radians(alpha_deg)
    ct, st = np.cos(theta_rad), np.sin(theta_rad)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [0,   sa,       ca,      d],
        [0,   0,        0,       1]
    ])

# --- Main Analytical Solver & Plotter ---
def plot_full_analytical_ik(t_x, t_y, t_z, roll, pitch, yaw):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # 1. Define Target Transformation Matrix
    R_target = rot_z(yaw) @ rot_y(pitch) @ rot_x(roll)
    P_target = np.array([t_x, t_y, t_z])
    
    # The Z-axis of R_target is our approach vector (3rd column)
    approach_vector = R_target[:, 2]
    
    # 2. Kinematic Decoupling: Find Wrist Center
    P_wc = P_target - (L6 * approach_vector)
    x_wc, y_wc, z_wc = P_wc
    
    # 3. Solve Position (Theta 1, 2, 3)
    theta1 = np.arctan2(y_wc, x_wc)
    
    r = np.sqrt(x_wc**2 + y_wc**2)
    s = z_wc - L1
    
    # Check Reachability
    D = (r**2 + s**2 - L2**2 - L4**2) / (2 * L2 * L4)
    if D < -1 or D > 1:
        ax.text(0, 0, 5, "TARGET OUT OF REACH", color='red', fontsize=20, weight='bold')
        ax.set_xlim([-6, 6]); ax.set_ylim([-6, 6]); ax.set_zlim([0, 8])
        plt.show()
        return

    # Solve Elbow (Theta 3)
    sin_t3 = D
    cos_t3 = -np.sqrt(1 - D**2) # Elbow up configuration
    theta3 = np.arctan2(sin_t3, cos_t3)
    
    # Solve Shoulder (Theta 2)
    A = L2 + L4 * sin_t3
    B = L4 * cos_t3
    theta2 = np.arctan2(B*r + A*s, A*r - B*s)
    
    # 4. Solve Orientation (Theta 4, 5, 6)
    # Calculate R0_3 (Rotation of first 3 joints)
    T0_1 = dh_matrix(theta1, L1, 0, 90)
    T1_2 = dh_matrix(theta2, 0, L2, 0)
    T2_3 = dh_matrix(theta3, 0, 0, 90)
    R0_3 = (T0_1 @ T1_2 @ T2_3)[:3, :3]
    
    # Calculate required wrist rotation: R3_6 = (R0_3)^T * R_target
    R3_6 = R0_3.T @ R_target
    
    # Extract Euler angles from spherical wrist matrix
    r13, r23, r33 = R3_6[0, 2], R3_6[1, 2], R3_6[2, 2]
    r31, r32 = R3_6[2, 0], R3_6[2, 1]
    
    theta5 = np.arctan2(np.sqrt(r13**2 + r23**2), r33)
    
    # Gimbal Lock check (if theta5 is close to 0)
    if np.abs(np.sin(theta5)) < 1e-6:
        theta4 = 0.0
        theta6 = np.arctan2(R3_6[1, 0], R3_6[0, 0])
    else:
        theta4 = np.arctan2(r23, r13)
        theta6 = np.arctan2(r32, -r31)
        
    # 5. Forward Kinematics to Plot Arm (Prove it works)
    T3_4 = dh_matrix(theta4, L4, 0, -90)
    T4_5 = dh_matrix(theta5, 0, 0, 90)
    T5_6 = dh_matrix(theta6, L6, 0, 0)
    
    T0 = np.eye(4)
    T1 = T0 @ T0_1
    T2 = T1 @ T1_2
    T3 = T2 @ T2_3
    T4 = T3 @ T3_4
    T5 = T4 @ T4_5
    T6 = T5 @ T5_6 # Final actual position
    
    joints = np.array([T0[:3,3], T1[:3,3], T2[:3,3], T3[:3,3], T4[:3,3], T5[:3,3], T6[:3,3]])
    ax.plot(joints[:,0], joints[:,1], joints[:,2], '-ko', linewidth=5, markersize=8, mfc='orange')
    
    # ==========================================
    # VISUALIZATION: ARROWS
    # ==========================================
    arrow_len = 1.5
    
    # 1. Plot Target Goal (Thick, faded arrows)
    ax.quiver(*P_target, *(R_target[:, 0] * arrow_len), color='red', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 1] * arrow_len), color='green', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 2] * arrow_len), color='blue', alpha=0.3, linewidth=6, label='Target (Faded)')

    # 2. Plot Actual Robot Tool (Thin, solid arrows)
    tool_pos = T6[:3, 3]
    R_actual = T6[:3, :3]
    ax.quiver(*tool_pos, *(R_actual[:, 0] * arrow_len), color='red', linewidth=2)
    ax.quiver(*tool_pos, *(R_actual[:, 1] * arrow_len), color='green', linewidth=2)
    ax.quiver(*tool_pos, *(R_actual[:, 2] * arrow_len), color='blue', linewidth=2, label='Actual Tool (Solid)')

    # Formatting
    ax.set_xlim([-6, 6]); ax.set_ylim([-6, 6]); ax.set_zlim([0, 8])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    
    title_str = (f'Solved! θ1:{np.degrees(theta1):.0f}°, θ2:{np.degrees(theta2):.0f}°, θ3:{np.degrees(theta3):.0f}°\n'
                 f'θ4:{np.degrees(theta4):.0f}°, θ5:{np.degrees(theta5):.0f}°, θ6:{np.degrees(theta6):.0f}°')
    ax.set_title(title_str)
    ax.legend()
    plt.show()

# Setup UI
interact(plot_full_analytical_ik,
         t_x=FloatSlider(value=4.0, min=-6, max=6, step=0.1, description='Target X'),
         t_y=FloatSlider(value=0.0, min=-6, max=6, step=0.1, description='Target Y'),
         t_z=FloatSlider(value=4.0, min=0, max=8, step=0.1, description='Target Z'),
         roll=FloatSlider(value=0.0, min=-180, max=180, step=5, description='Roll (X)'),
         pitch=FloatSlider(value=0.0, min=-180, max=180, step=5, description='Pitch (Y)'),
         yaw=FloatSlider(value=0.0, min=-180, max=180, step=5, description='Yaw (Z)'))

interactive(children=(FloatSlider(value=4.0, description='Target X', max=6.0, min=-6.0), FloatSlider(value=0.0…

<function __main__.plot_full_analytical_ik(t_x, t_y, t_z, roll, pitch, yaw)>

### The Failure of Analytical Math: Gimbal Lock
Analytical Inverse Kinematics relies on Euler angles (Roll, Pitch, Yaw) to describe 3D orientation. However, Euler angles have a fatal mathematical flaw known as **Gimbal Lock**.

Gimbal Lock occurs when the middle rotation (Pitch) aligns exactly at $90^\circ$ or $-90^\circ$. Let's look at what happens to our $3 \times 3$ Rotation Matrix ($R = R_z \cdot R_y \cdot R_x$) when Pitch ($\beta$) is exactly $90^\circ$:

$$R_{gimbal} = \begin{bmatrix} 0 & \sin(\alpha-\gamma) & \cos(\alpha-\gamma) \\ 0 & \cos(\alpha-\gamma) & -\sin(\alpha-\gamma) \\ -1 & 0 & 0 \end{bmatrix}$$

Notice how the Roll ($\alpha$) and Yaw ($\gamma$) angles have mathematically merged into a single term: $(\alpha-\gamma)$! 

**The Physical Result:** The robot has lost a degree of freedom. Rotating the base (Yaw) and rotating the wrist (Roll) now do the exact same physical thing. Because two axes have perfectly aligned, the analytical solver encounters infinite possible solutions and completely crashes.

[To understand more about Gimbal lock visually visit](https://arm-kinematics-webapp.vercel.app/#/gimbal-lock)

### The Solution to Redundancy: The Null Space
Analytical methods require exactly 6 motors to place a tool in 6-DOF space (X, Y, Z, Roll, Pitch, Yaw). If you add a 7th motor, analytical geometry fails because there are now **infinite** ways to reach the target. 

Numerical solvers (using the Jacobian Pseudo-Inverse) thrive here. They allow us to utilize the **Null Space**.
The Null Space is the set of all joint movements that result in zero velocity at the end effector. 

$$J \cdot \dot{\theta}_{null} = 0$$

This means we can mathematically command the robot to swing its elbow, dodge obstacles, or avoid joint limits, all while keeping the tool perfectly pinned to the target! Let's visualize this using a 3-link planar arm reaching for a 2D coordinate. (3 motors for a 2D target = 1 redundant degree of freedom).

[Null space visual element](https://arm-kinematics-webapp.vercel.app/#/null-space)

In [25]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# 3-Link Planar Arm (Redundant for a 2D position target)
L1, L2, L3 = 2.0, 2.0, 2.0

def plot_null_space(target_x, target_y, null_space_motion):
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 1. Check max reachability
    max_reach = L1 + L2 + L3
    dist_to_target = np.sqrt(target_x**2 + target_y**2)
    
    if dist_to_target > max_reach:
        ax.text(0, 0, "TARGET OUT OF REACH", color='red', fontsize=16, ha='center')
        ax.set_xlim([-7, 7]); ax.set_ylim([-7, 7])
        plt.show()
        return

    # 2. Inject Null Space Motion (Driving Theta 1 arbitrarily)
    theta1 = np.radians(null_space_motion)
    
    # Calculate position of Joint 1
    j1_x = L1 * np.cos(theta1)
    j1_y = L1 * np.sin(theta1)
    
    # 3. Solve 2-Link IK for the remaining distance (Theta 2 and Theta 3)
    # The new target for the remaining 2 links is the actual target minus j1 position
    rx = target_x - j1_x
    ry = target_y - j1_y
    r_sq = rx**2 + ry**2
    
    # Check if the remaining distance is reachable by Links 2 and 3
    cos_t3 = (r_sq - L2**2 - L3**2) / (2 * L2 * L3)
    
    if cos_t3 < -1 or cos_t3 > 1:
        # The null space motion forced the arm into an impossible pose
        ax.scatter(target_x, target_y, color='red', s=200, marker='*')
        ax.plot([0, j1_x], [0, j1_y], color='blue', linewidth=4)
        ax.text(0, 4, "Null Space limit reached.\nCannot maintain target constraint.", color='red', fontsize=12, ha='center')
    else:
        # Solve the remaining angles
        sin_t3 = -np.sqrt(1 - cos_t3**2) # Elbow up
        theta3_local = np.arctan2(sin_t3, cos_t3)
        
        theta2_local = np.arctan2(ry, rx) - np.arctan2(L3 * sin_t3, L2 + L3 * cos_t3)
        
        # Absolute angles for FK plotting
        theta2 = theta2_local
        theta3 = theta2 + theta3_local
        
        j2_x = j1_x + L2 * np.cos(theta2)
        j2_y = j1_y + L2 * np.sin(theta2)
        
        tool_x = j2_x + L3 * np.cos(theta3)
        tool_y = j2_y + L3 * np.sin(theta3)
        
        # Plot Arm
        x_coords = [0, j1_x, j2_x, tool_x]
        y_coords = [0, j1_y, j2_y, tool_y]
        ax.plot(x_coords, y_coords, '-ko', linewidth=5, markersize=8, mfc='orange')
        
        # Plot Target
        ax.scatter(target_x, target_y, color='green', s=300, marker='*', zorder=5, label='Fixed Target')

    # Formatting
    ax.set_xlim([-7, 7]); ax.set_ylim([-7, 7])
    ax.grid(True)
    ax.set_title(f'Null Space Self-Motion\nTarget locked at ({target_x}, {target_y})')
    ax.legend()
    plt.show()

# Setup UI
interact(plot_null_space,
         target_x=FloatSlider(value=3.0, min=-6, max=6, step=0.5, description='Target X'),
         target_y=FloatSlider(value=3.0, min=-6, max=6, step=0.5, description='Target Y'),
         null_space_motion=FloatSlider(value=0, min=-180, max=180, step=2, description='Null Space Twist'))

interactive(children=(FloatSlider(value=3.0, description='Target X', max=6.0, min=-6.0, step=0.5), FloatSlider…

<function __main__.plot_null_space(target_x, target_y, null_space_motion)>

### The Mathematical Engine: The Geometric Jacobian
To solve Inverse Kinematics numerically, we need to map the velocity of our joints ($\dot{\theta}$) to the velocity of the tool ($\dot{X}$). This mapping is the **Jacobian Matrix ($J$)**.

$$\begin{bmatrix} v_x \\ v_y \\ v_z \\ \omega_x \\ \omega_y \\ \omega_z \end{bmatrix} = J \begin{bmatrix} \dot{\theta_1} \\ \dot{\theta_2} \\ \dot{\theta_3} \\ \dot{\theta_4} \\ \dot{\theta_5} \\ \dot{\theta_6} \end{bmatrix}$$

For a 6-DOF arm, $J$ is a $6 \times 6$ matrix. Each column of the Jacobian corresponds to one joint, and it tells us exactly how moving that *single* joint affects the tool.

Instead of doing insane partial derivatives, we can build the Jacobian geometrically! Every column $i$ is split into two parts: Linear Velocity ($v$) and Angular Velocity ($\omega$).

#### 1. Angular Velocity ($\omega$)
For a revolute joint, the angular velocity it applies to the tool is simply the axis it spins around! We already know this: it's the $Z$-axis of the previous frame.
**$$\omega_i = Z_{i-1}$$**

#### 2. Linear Velocity ($v$)
If you spin a joint, the tool moves in an arc. The direction of that movement is perpendicular to both the axis of rotation ($Z_{i-1}$) and the vector pointing from the joint to the tool ($P_{tool} - P_{i-1}$). We find this using the Cross Product:
**$$v_i = Z_{i-1} \times (P_{tool} - P_{i-1})$$**

#### Building the Matrix
By stacking these together, the $i$-th column of our Jacobian is simply:
$$J_i = \begin{bmatrix} Z_{i-1} \times (P_{tool} - P_{i-1}) \\ Z_{i-1} \end{bmatrix}$$

Because our Forward Kinematics matrix ($T$) already gives us the $Z$-axis (3rd column) and the Position (4th column) for every joint, calculating the exact mathematical Jacobian becomes incredibly easy!

In [26]:
import numpy as np

# Physical Link Lengths (matching our previous tables)
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# Classical DH Function
def classical_dh_matrix(theta_deg, d, a, alpha_deg):
    theta = np.radians(theta_deg)
    alpha = np.radians(alpha_deg)
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [0,   sa,       ca,      d],
        [0,   0,        0,       1]
    ])

def print_geometric_jacobian(t1, t2, t3, t4, t5, t6):
    # 1. Calculate all transformation matrices
    T0_1 = classical_dh_matrix(t1, L1,  0,   90)
    T1_2 = classical_dh_matrix(t2,  0, L2,    0)
    T2_3 = classical_dh_matrix(t3,  0,  0,   90)
    T3_4 = classical_dh_matrix(t4, L4,  0,  -90)
    T4_5 = classical_dh_matrix(t5,  0,  0,   90)
    T5_6 = classical_dh_matrix(t6, L6,  0,    0)

    # Chain them to get the absolute transformations from Base (0)
    T0 = np.eye(4)
    T1 = T0 @ T0_1
    T2 = T1 @ T1_2
    T3 = T2 @ T2_3
    T4 = T3 @ T3_4
    T5 = T4 @ T4_5
    T6 = T5 @ T5_6 # Final End Effector

    # Store matrices in a list for easy looping
    transforms = [T0, T1, T2, T3, T4, T5]
    
    # 2. Extract the Tool Position
    P_tool = T6[:3, 3]

    # 3. Build the 6x6 Jacobian Matrix
    J = np.zeros((6, 6))

    for i in range(6):
        # Extract Z-axis of the previous frame (3rd column, first 3 rows)
        Z_prev = transforms[i][:3, 2]
        
        # Extract Position of the previous frame (4th column, first 3 rows)
        P_prev = transforms[i][:3, 3]
        
        # Linear Velocity (Cross Product)
        v = np.cross(Z_prev, (P_tool - P_prev))
        
        # Angular Velocity (Just the Z-axis)
        w = Z_prev
        
        # Stack them into the i-th column of J
        J[:3, i] = v
        J[3:, i] = w

    # 4. Print the result beautifully formatted
    np.set_printoptions(precision=3, suppress=True)
    print("--- Tool Position (X, Y, Z) ---")
    print(P_tool)
    print("\n--- The 6x6 Geometric Jacobian Matrix ---")
    print("      θ1      θ2      θ3      θ4      θ5      θ6")
    print("-" * 55)
    
    labels = ['vx', 'vy', 'vz', 'wx', 'wy', 'wz']
    for row_idx in range(6):
        row_str = "  ".join([f"{val:6.2f}" for val in J[row_idx, :]])
        print(f"{labels[row_idx]} | {row_str}")
        if row_idx == 2:
            print("-" * 55) # Separator between linear and angular

# Test the function with some arbitrary angles
print_geometric_jacobian(t1=45, t2=30, t3=-45, t4=0, t5=90, t6=0)

--- Tool Position (X, Y, Z) ---
[2.063 2.063 0.826]

--- The 6x6 Geometric Jacobian Matrix ---
      θ1      θ2      θ3      θ4      θ5      θ6
-------------------------------------------------------
vx |  -2.06    0.83    1.89    0.71    0.18    0.00
vy |   2.06    0.83    1.89   -0.71    0.18    0.00
vz |   0.00    2.92    0.32    0.00    0.97    0.00
-------------------------------------------------------
wx |   0.00    0.71    0.71   -0.18    0.71    0.68
wy |   0.00   -0.71   -0.71   -0.18   -0.71    0.68
wz |   1.00    0.00    0.00   -0.97    0.00   -0.26


### The Analytical Jacobian (Calculus Approach)
We just built the **Geometric Jacobian** using physical vectors and cross products. However, if you look in most calculus textbooks, the Jacobian is defined strictly through partial derivatives. This is called the **Analytical Jacobian** ($J_A$).

If our Forward Kinematics gives us a set of non-linear equations for our tool's position ($X, Y, Z$) and orientation using Euler angles ($\alpha, \beta, \gamma$), we can define the pose as a single vector $P$:

$$P = \begin{bmatrix} X(\theta) \\ Y(\theta) \\ Z(\theta) \\ \alpha(\theta) \\ \beta(\theta) \\ \gamma(\theta) \end{bmatrix}$$

To find how a change in joint angles ($\dot{\theta}$) affects the pose ($\dot{P}$), we must take the **partial derivative** of every single equation with respect to every single joint. This creates our $6 \times 6$ matrix:

$$J_A = \begin{bmatrix} \frac{\partial X}{\partial \theta_1} & \frac{\partial X}{\partial \theta_2} & \frac{\partial X}{\partial \theta_3} & \frac{\partial X}{\partial \theta_4} & \frac{\partial X}{\partial \theta_5} & \frac{\partial X}{\partial \theta_6} \\ \frac{\partial Y}{\partial \theta_1} & \frac{\partial Y}{\partial \theta_2} & \frac{\partial Y}{\partial \theta_3} & \frac{\partial Y}{\partial \theta_4} & \frac{\partial Y}{\partial \theta_5} & \frac{\partial Y}{\partial \theta_6} \\ \frac{\partial Z}{\partial \theta_1} & \frac{\partial Z}{\partial \theta_2} & \frac{\partial Z}{\partial \theta_3} & \frac{\partial Z}{\partial \theta_4} & \frac{\partial Z}{\partial \theta_5} & \frac{\partial Z}{\partial \theta_6} \\ \frac{\partial \alpha}{\partial \theta_1} & \frac{\partial \alpha}{\partial \theta_2} & \frac{\partial \alpha}{\partial \theta_3} & \frac{\partial \alpha}{\partial \theta_4} & \frac{\partial \alpha}{\partial \theta_5} & \frac{\partial \alpha}{\partial \theta_6} \\ \frac{\partial \beta}{\partial \theta_1} & \frac{\partial \beta}{\partial \theta_2} & \frac{\partial \beta}{\partial \theta_3} & \frac{\partial \beta}{\partial \theta_4} & \frac{\partial \beta}{\partial \theta_5} & \frac{\partial \beta}{\partial \theta_6} \\ \frac{\partial \gamma}{\partial \theta_1} & \frac{\partial \gamma}{\partial \theta_2} & \frac{\partial \gamma}{\partial \theta_3} & \frac{\partial \gamma}{\partial \theta_4} & \frac{\partial \gamma}{\partial \theta_5} & \frac{\partial \gamma}{\partial \theta_6} \end{bmatrix}$$

#### Geometric vs. Analytical: The Crucial Difference
The top half of both matrices (the linear velocities $v_x, v_y, v_z$) are mathematically identical. 

However, the bottom half is fundamentally different:
* The **Geometric Jacobian** outputs the true 3D angular velocity vector ($\omega_x, \omega_y, \omega_z$).
* The **Analytical Jacobian** outputs the rate of change of your specific Euler angles ($\dot{\alpha}, \dot{\beta}, \dot{\gamma}$).

Because Euler angles suffer from **Gimbal Lock**, taking the partial derivatives of those angles means the Analytical Jacobian will mathematically crash if the robot enters a singularity. By using the Geometric Jacobian instead, we avoid differentiating Euler angles, keeping our numerical solvers much more stable!

### The Math of Redundancy: The Null Space Projector
We know the Jacobian Pseudo-Inverse ($J^+$) solves the Inverse Kinematics problem by finding the *Minimum Norm* solution. It moves the tool to the target using the absolute smallest possible joint rotations.

But what if we *want* the joints to move more? What if an obstacle is in the way, or a joint is about to hit its mechanical limit? 

To control the internal joints without moving the tool, we use the **Null Space Projection Matrix**. The master equation for modern redundant robotic control looks like this:

$$\dot{\theta} = J^+ \dot{X} + (I - J^+ J) \dot{\theta}_{sec}$$

Let's break this down into its two distinct halves:

#### 1. The Primary Task: $J^+ \dot{X}$
This is the standard Pseudo-Inverse step. It calculates the exact joint velocities ($\dot{\theta}$) required to move the tool toward the target ($\dot{X}$). It has absolute priority.

#### 2. The Secondary Task: $(I - J^+ J) \dot{\theta}_{sec}$
This is where the magic happens. 
* $\dot{\theta}_{sec}$ is our secondary goal. For example, it could be a command saying, *"Pull the base joint towards $90^\circ$."*
* $I$ is the Identity Matrix.
* **$(I - J^+ J)$** is the **Null Space Projector**.

If we just added our secondary goal directly to the motors, the arm would swing wildly and pull the tool off the target. Multiplying our secondary goal by the Null Space Projector acts as a mathematical filter. It strips away any part of the command that would cause the tool to move, leaving *only* the joint velocities that cause self-motion!

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Physical links
L1, L2, L3 = 2.0, 2.0, 2.0

# 3-Link Forward Kinematics (Returns 2D X, Y position)
def get_fk_2d(q):
    x = L1*np.cos(q[0]) + L2*np.cos(q[0]+q[1]) + L3*np.cos(q[0]+q[1]+q[2])
    y = L1*np.sin(q[0]) + L2*np.sin(q[0]+q[1]) + L3*np.sin(q[0]+q[1]+q[2])
    return np.array([x, y])

# 2x3 Jacobian using Finite Difference
def get_jacobian_2d(q):
    epsilon = 1e-5
    J = np.zeros((2, 3))
    pos_current = get_fk_2d(q)
    
    for i in range(3):
        q_wiggle = q.copy()
        q_wiggle[i] += epsilon
        pos_wiggled = get_fk_2d(q_wiggle)
        J[:, i] = (pos_wiggled - pos_current) / epsilon
    return J

# Store last known good angles to keep solver stable across slider movements
current_q = np.array([np.pi/4, -np.pi/4, -np.pi/4])

def plot_null_space_jacobian(target_x, target_y, desired_base_angle):
    global current_q
    q = current_q.copy()
    target = np.array([target_x, target_y])
    
    max_iterations = 50
    learning_rate = 0.2
    
    # Target for our secondary task (in radians)
    theta_sec_target = np.radians(desired_base_angle)
    
    for _ in range(max_iterations):
        pos = get_fk_2d(q)
        error = target - pos
        
        J = get_jacobian_2d(q)
        
        # Calculate Pseudo-Inverse
        J_pinv = np.linalg.pinv(J)
        
        # 1. PRIMARY TASK: Move tool to target
        primary_step = J_pinv @ error
        
        # 2. SECONDARY TASK: Pull base joint to desired angle
        # We create a velocity vector that only pushes joint 0
        q_sec_velocity = np.array([theta_sec_target - q[0], 0.0, 0.0])
        
        # 3. NULL SPACE PROJECTOR: (I - J^+ * J)
        I = np.eye(3)
        N = I - (J_pinv @ J)
        
        # Filter the secondary task through the Null Space
        secondary_step = N @ q_sec_velocity
        
        # 4. COMBINE AND UPDATE
        # We add the filtered secondary step to the primary step
        q = q + learning_rate * (primary_step + secondary_step)
        
    current_q = q
    
    # --- Plotting Logic ---
    j1_x, j1_y = L1*np.cos(q[0]), L1*np.sin(q[0])
    j2_x, j2_y = j1_x + L2*np.cos(q[0]+q[1]), j1_y + L2*np.sin(q[0]+q[1])
    tool_x, tool_y = get_fk_2d(q)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot target and arm
    ax.scatter(target_x, target_y, color='green', s=300, marker='*', zorder=5, label='Primary Target')
    x_coords, y_coords = [0, j1_x, j2_x, tool_x], [0, j1_y, j2_y, tool_y]
    ax.plot(x_coords, y_coords, '-ko', linewidth=5, markersize=8, mfc='orange')
    
    # Formatting
    ax.set_xlim([-7, 7]); ax.set_ylim([-7, 7]); ax.grid(True)
    dist_error = np.linalg.norm(target - np.array([tool_x, tool_y]))
    
    if dist_error > 0.1:
        ax.set_title(f"Target Unreachable! Tool Error: {dist_error:.2f}", color='red')
    else:
        ax.set_title(f"Null Space Projection Active\nTool Error: {dist_error:.4f} | Base Angle: {np.degrees(q[0]):.1f}°")
        
    ax.legend()
    plt.show()

# Setup UI
interact(plot_null_space_jacobian,
         target_x=FloatSlider(value=3.0, min=-5, max=5, step=0.1, description='Target X'),
         target_y=FloatSlider(value=3.0, min=-5, max=5, step=0.1, description='Target Y'),
         desired_base_angle=FloatSlider(value=45, min=-180, max=180, step=5, description='Sec. Task: Base °'))

interactive(children=(FloatSlider(value=3.0, description='Target X', max=5.0, min=-5.0), FloatSlider(value=3.0…

<function __main__.plot_null_space_jacobian(target_x, target_y, desired_base_angle)>

### The Full 6-DOF Numerical Solver (Step-by-Step)
We are now ready to build a professional-grade Inverse Kinematics solver. We will use our **Geometric Jacobian** and the **Pseudo-Inverse** to iteratively pull our 6-DOF arm to a 6-DOF target (X, Y, Z + Roll, Pitch, Yaw).

Here is the step-by-step control loop we will program:

#### Step 1: Where are we? (Forward Kinematics)
We calculate our current $4 \times 4$ transformation matrix ($T_{current}$) based on our current joint angles.

#### Step 2: Calculate the Error ($E$)
We need a $6 \times 1$ error vector to match our $6 \times 6$ Jacobian. We split this into two parts:
* **Position Error ($e_p$):** Simple subtraction. $e_p = P_{target} - P_{current}$
* **Orientation Error ($e_o$):** We *cannot* just subtract Euler angles, or we risk Gimbal Lock! Instead, we compare the actual 3D directional arrows (X, Y, Z axes) of our tool against the target's arrows using cross-products. 
  $$e_o = \frac{1}{2} (X_c \times X_{target} + Y_c \times Y_{target} + Z_c \times Z_{target})$$

We stack these into our final error vector: $E = \begin{bmatrix} e_p \\ e_o \end{bmatrix}$

#### Step 3: Compute the Geometric Jacobian ($J$)
We use the cross-product method we derived earlier to build the $6 \times 6$ matrix mapping joint velocities to tool velocities at this exact configuration.

#### Step 4: The Pseudo-Inverse Step
We calculate the change in joint angles required to crush our error using the Pseudo-Inverse.
$$\Delta \theta = J^+ \cdot E$$

#### Step 5: Update and Repeat
We add a small fraction of $\Delta \theta$ to our current motors ($\theta_{new} = \theta_{old} + \alpha \Delta \theta$) and repeat Steps 1-5 until the error drops to zero!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Physical Link Lengths
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# --- Helper Functions ---
def rot_x(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def rot_y(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rot_z(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def dh_matrix(theta_rad, d, a, alpha_deg):
    alpha = np.radians(alpha_deg)
    ct, st = np.cos(theta_rad), np.sin(theta_rad)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [0,   sa,       ca,      d],
        [0,   0,        0,       1]
    ])

# Global variable to store last known angles to keep the solver continuous
current_q = np.zeros(6)

def plot_full_numerical_ik(t_x, t_y, t_z, roll, pitch, yaw):
    global current_q
    q = current_q.copy()
    
    # Define Target Position and Rotation
    P_target = np.array([t_x, t_y, t_z])
    R_target = rot_z(yaw) @ rot_y(pitch) @ rot_x(roll)
    
    max_iterations = 100
    learning_rate = 0.5
    tolerance = 1e-3
    
    # --- THE NUMERICAL IK LOOP ---
    for _ in range(max_iterations):
        # 1. Forward Kinematics
        T0_1 = dh_matrix(q[0], L1, 0, 90)
        T1_2 = dh_matrix(q[1], 0, L2, 0)
        T2_3 = dh_matrix(q[2], 0, 0, 90)
        T3_4 = dh_matrix(q[3], L4, 0, -90)
        T4_5 = dh_matrix(q[4], 0, 0, 90)
        T5_6 = dh_matrix(q[5], L6, 0, 0)
        
        transforms = [np.eye(4)]
        for T_link in [T0_1, T1_2, T2_3, T3_4, T4_5, T5_6]:
            transforms.append(transforms[-1] @ T_link)
            
        T_tool = transforms[-1]
        P_curr = T_tool[:3, 3]
        R_curr = T_tool[:3, :3]
        
        # 2. Calculate Error (Position + Orientation)
        # Position Error
        e_p = P_target - P_curr
        
        # Orientation Error (Cross-product method to avoid Gimbal Lock)
        x_c, y_c, z_c = R_curr[:, 0], R_curr[:, 1], R_curr[:, 2]
        x_d, y_d, z_d = R_target[:, 0], R_target[:, 1], R_target[:, 2]
        e_o = 0.5 * (np.cross(x_c, x_d) + np.cross(y_c, y_d) + np.cross(z_c, z_d))
        
        # Stack into 6x1 vector
        E = np.hstack((e_p, e_o))
        
        if np.linalg.norm(E) < tolerance:
            break
            
        # 3. Geometric Jacobian
        J = np.zeros((6, 6))
        for i in range(6):
            Z_prev = transforms[i][:3, 2]
            P_prev = transforms[i][:3, 3]
            J[:3, i] = np.cross(Z_prev, (P_curr - P_prev)) # Linear velocity
            J[3:, i] = Z_prev                              # Angular velocity
            
        # 4. Pseudo-Inverse and Update
        J_pinv = np.linalg.pinv(J)
        delta_q = J_pinv @ E
        q = q + (learning_rate * delta_q)
        
    current_q = q
    
    # --- PLOTTING ---
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot Arm Links
    joints = np.array([T[:3, 3] for T in transforms])
    ax.plot(joints[:,0], joints[:,1], joints[:,2], '-ko', linewidth=5, markersize=8, mfc='orange')
    
    # Plot Target Orientation (Faded Arrows)
    arrow_len = 1.5
    ax.quiver(*P_target, *(R_target[:, 0] * arrow_len), color='red', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 1] * arrow_len), color='green', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 2] * arrow_len), color='blue', alpha=0.3, linewidth=6, label='Target (Faded)')

    # Plot Actual Tool Orientation (Solid Arrows)
    ax.quiver(*P_curr, *(R_curr[:, 0] * arrow_len), color='red', linewidth=2)
    ax.quiver(*P_curr, *(R_curr[:, 1] * arrow_len), color='green', linewidth=2)
    ax.quiver(*P_curr, *(R_curr[:, 2] * arrow_len), color='blue', linewidth=2, label='Tool (Solid)')

    # Formatting
    ax.set_xlim([-6, 6]); ax.set_ylim([-6, 6]); ax.set_zlim([0, 8])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    
    total_error = np.linalg.norm(E)
    title_str = (f'6-DOF Numerical IK Converged\n'
                 f'Total Error: {total_error:.5f}')
    if total_error > 0.1:
        title_str = f"Target Out of Reach or Stuck in Local Minima\nError: {total_error:.2f}"
        
    ax.set_title(title_str)
    ax.legend()
    plt.show()

# Setup UI
interact(plot_full_numerical_ik,
         t_x=FloatSlider(value=4.0, min=-6, max=6, step=0.2, description='Target X'),
         t_y=FloatSlider(value=0.0, min=-6, max=6, step=0.2, description='Target Y'),
         t_z=FloatSlider(value=4.0, min=0, max=8, step=0.2, description='Target Z'),
         roll=FloatSlider(value=0.0, min=-180, max=180, step=10, description='Roll (X)'),
         pitch=FloatSlider(value=0.0, min=-180, max=180, step=10, description='Pitch (Y)'),
         yaw=FloatSlider(value=0.0, min=-180, max=180, step=10, description='Yaw (Z)'))

interactive(children=(FloatSlider(value=4.0, description='Target X', max=6.0, min=-6.0, step=0.2), FloatSlider…

<function __main__.plot_full_numerical_ik(t_x, t_y, t_z, roll, pitch, yaw)>

### The Singularity Trap: Why the Math Explodes
If you push the target out of the robot's physical reach, or if the arm sweeps through a singularity (like Gimbal Lock or a fully locked elbow), you might notice the arm violently "teleports" to a random, broken configuration. 

**This is not a bug in the code; it is a mathematical breakdown of the Jacobian.**

When a robot loses a degree of freedom, a row or column in the Jacobian matrix effectively zeros out. The matrix loses its "full rank." When we calculate the Pseudo-Inverse ($J^+$) of a collapsing matrix, the math panics. It attempts to divide by near-zero, resulting in **massive, near-infinite joint velocity commands**. 

It essentially tells the motors: *"To fix this 1-millimeter error, spin 10,000 degrees instantly!"* Because our basic loop blindly adds this massive $\Delta \theta$ to our joints, the arm explodes. In the real world, this snaps gears or triggers a violent Emergency Stop.

### The Solution: Damped Least Squares (DLS)
To stop the arm from destroying itself, we must put a mathematical leash on the solver. While we could use a simple speed limit (Velocity Clamping), the industry standard for professional robotic controllers is **Damped Least Squares (DLS)**, often tied to the Levenberg-Marquardt algorithm.

Instead of just calculating the standard Pseudo-Inverse, DLS modifies the inversion by injecting a **damping factor** ($\lambda$):

$$\Delta \theta = (J^T J + \lambda^2 I)^{-1} J^T E$$

**How it works:**
* $I$ is the Identity Matrix.
* $\lambda$ acts as a mathematical "shock absorber."

When the robot is moving safely in its workspace, $\lambda$ is negligible, and the equation acts exactly like our old Pseudo-Inverse. But as the robot approaches a singularity and $J$ begins to collapse, the $\lambda^2 I$ term acts as a safety floor. It prevents the denominator from hitting zero. 

The result? The robot will gracefully slow down and stretch safely toward an unreachable target instead of violently crashing!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Physical Link Lengths
L1, L2, L4, L6 = 2.0, 3.0, 2.5, 1.0

# --- Helper Functions ---
def rot_x(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def rot_y(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rot_z(deg):
    rad = np.radians(deg)
    c, s = np.cos(rad), np.sin(rad)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def dh_matrix(theta_rad, d, a, alpha_deg):
    alpha = np.radians(alpha_deg)
    ct, st = np.cos(theta_rad), np.sin(theta_rad)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [0,   sa,       ca,      d],
        [0,   0,        0,       1]
    ])

# Global variable to store last known angles
current_q = np.zeros(6)

def plot_dls_numerical_ik(t_x, t_y, t_z, roll, pitch, yaw):
    global current_q
    q = current_q.copy()
    
    P_target = np.array([t_x, t_y, t_z])
    R_target = rot_z(yaw) @ rot_y(pitch) @ rot_x(roll)
    
    max_iterations = 100
    learning_rate = 0.5
    tolerance = 1e-3
    
    # --- DAMPING FACTOR ---
    # The higher the lambda, the more sluggish the arm acts near singularities.
    # 0.1 is a standard starting point for smooth operation.
    lambda_damping = 0.1 
    I = np.eye(6)
    
    # --- THE NUMERICAL IK LOOP ---
    for _ in range(max_iterations):
        # 1. Forward Kinematics
        T0_1 = dh_matrix(q[0], L1, 0, 90)
        T1_2 = dh_matrix(q[1], 0, L2, 0)
        T2_3 = dh_matrix(q[2], 0, 0, 90)
        T3_4 = dh_matrix(q[3], L4, 0, -90)
        T4_5 = dh_matrix(q[4], 0, 0, 90)
        T5_6 = dh_matrix(q[5], L6, 0, 0)
        
        transforms = [np.eye(4)]
        for T_link in [T0_1, T1_2, T2_3, T3_4, T4_5, T5_6]:
            transforms.append(transforms[-1] @ T_link)
            
        T_tool = transforms[-1]
        P_curr = T_tool[:3, 3]
        R_curr = T_tool[:3, :3]
        
        # 2. Calculate Error (Position + Orientation)
        e_p = P_target - P_curr
        x_c, y_c, z_c = R_curr[:, 0], R_curr[:, 1], R_curr[:, 2]
        x_d, y_d, z_d = R_target[:, 0], R_target[:, 1], R_target[:, 2]
        e_o = 0.5 * (np.cross(x_c, x_d) + np.cross(y_c, y_d) + np.cross(z_c, z_d))
        E = np.hstack((e_p, e_o))
        
        if np.linalg.norm(E) < tolerance:
            break
            
        # 3. Geometric Jacobian
        J = np.zeros((6, 6))
        for i in range(6):
            Z_prev = transforms[i][:3, 2]
            P_prev = transforms[i][:3, 3]
            J[:3, i] = np.cross(Z_prev, (P_curr - P_prev))
            J[3:, i] = Z_prev
            
        # 4. DAMPED LEAST SQUARES (The Fix)
        # J_dls = (J^T * J + lambda^2 * I)^-1 * J^T
        J_dls = np.linalg.inv(J.T @ J + (lambda_damping**2) * I) @ J.T
        delta_q = J_dls @ E
        
        # Safety clamp just to ensure we never over-rotate per frame
        delta_q = np.clip(delta_q, -0.2, 0.2) 
        
        q = q + (learning_rate * delta_q)
        
    current_q = q
    
    # --- PLOTTING ---
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    joints = np.array([T[:3, 3] for T in transforms])
    ax.plot(joints[:,0], joints[:,1], joints[:,2], '-ko', linewidth=5, markersize=8, mfc='orange')
    
    arrow_len = 1.5
    ax.quiver(*P_target, *(R_target[:, 0] * arrow_len), color='red', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 1] * arrow_len), color='green', alpha=0.3, linewidth=6)
    ax.quiver(*P_target, *(R_target[:, 2] * arrow_len), color='blue', alpha=0.3, linewidth=6, label='Target')

    ax.quiver(*P_curr, *(R_curr[:, 0] * arrow_len), color='red', linewidth=2)
    ax.quiver(*P_curr, *(R_curr[:, 1] * arrow_len), color='green', linewidth=2)
    ax.quiver(*P_curr, *(R_curr[:, 2] * arrow_len), color='blue', linewidth=2, label='Actual')

    ax.set_xlim([-6, 6]); ax.set_ylim([-6, 6]); ax.set_zlim([0, 8])
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    
    total_error = np.linalg.norm(E)
    if total_error > 0.1:
        ax.set_title(f"DLS Safety Engaged | Out of Reach\nError: {total_error:.2f}", color='red')
    else:
        ax.set_title(f"Target Locked\nError: {total_error:.5f}")
        
    ax.legend()
    plt.show()

interact(plot_dls_numerical_ik,
         t_x=FloatSlider(value=4.0, min=-6, max=6, step=0.5, description='Target X'),
         t_y=FloatSlider(value=0.0, min=-6, max=6, step=0.5, description='Target Y'),
         t_z=FloatSlider(value=4.0, min=0, max=8, step=0.5, description='Target Z'),
         roll=FloatSlider(value=0.0, min=-180, max=180, step=15, description='Roll (X)'),
         pitch=FloatSlider(value=0.0, min=-180, max=180, step=15, description='Pitch (Y)'),
         yaw=FloatSlider(value=0.0, min=-180, max=180, step=15, description='Yaw (Z)'))

interactive(children=(FloatSlider(value=4.0, description='Target X', max=6.0, min=-6.0, step=0.5), FloatSlider…

<function __main__.plot_dls_numerical_ik(t_x, t_y, t_z, roll, pitch, yaw)>

### The Flaw in the Matrix and the DLS Band-Aid
While Damped Least Squares (DLS) saves the robot from tearing itself apart, it is ultimately a mathematical band-aid. By artificially dampening the velocities using the $\lambda$ factor, the robot **sacrifices accuracy** to save its hardware. If you track the tool during a DLS maneuver near a singularity, you will see it physically drift off the perfect target path.

Furthermore, the very foundation of our math—the $4 \times 4$ Homogeneous Transformation Matrix and Euler angles—is fundamentally flawed:
* **Gimbal Lock:** Any system relying on 3-axis sequential rotations will inevitably hit blind spots.
* **Poor Interpolation:** If you tell a robot to smoothly rotate its wrist from Pose A to Pose B using matrices, the math often forces the tool to swing in a bizarre, unnatural arc.

To achieve flawless, singularity-free path planning, state-of-the-art robotics controllers ditch matrices entirely.

---

### 19. The Ultimate Solution: Dual Quaternions
To fix Gimbal Lock, aerospace and robotics engineers use **Quaternions**. A quaternion ($q$) uses four numbers (one real, three imaginary) to describe a 3D rotation, completely eliminating the singularities caused by 3-number Euler angles. 
    
$$q = q_w + q_x i + q_y j + q_z k$$

However, standard quaternions *only* handle rotation. To calculate a robot's kinematics, we need both rotation and translation (position). To bind them together into a single mathematical object, we use **Dual Quaternions** ($\hat{q}$). 

A Dual Quaternion consists of a "Real" part ($q_r$, which handles rotation) and a "Dual" part ($q_d$, which handles translation). They are linked by a special dual unit ($\epsilon$), defined by the rule that $\epsilon^2 = 0$ (but $\epsilon \neq 0$).

$$\hat{q} = q_r + \epsilon q_d$$

#### Why Dual Quaternions Rule Modern Robotics:
1. **Zero Singularities:** Because they do not rely on sequential axes, Dual Quaternions are completely immune to Gimbal Lock.
2. **Shortest-Path Interpolation:** Using an algorithm called ScLERP (Screw Linear Interpolation), Dual Quaternions mathematically guarantee the tool moves from point A to point B along the absolute smoothest, shortest possible arc in 3D space.
3. **Computational Efficiency:** Multiplying Dual Quaternions requires significantly fewer floating-point operations than multiplying massive $4 \times 4$ matrices, allowing the robot's control loop to run much faster.

#### Extracting the Data
If a Dual Quaternion represents your robot's end-effector, pulling the physical coordinates out of it is incredibly elegant:
* **Rotation:** is exactly the real quaternion: $q_r$
* **Translation ($X, Y, Z$):** is extracted by multiplying the dual part by the conjugate of the real part ($q_r^*$):

$$t = 2 q_d q_r^*$$

By rewriting our Jacobian to map joint velocities directly to Dual Quaternion rates of change, we create an Inverse Kinematics solver that is fast, mathematically bulletproof, and capable of gracefully navigating any workspace.

### Visualizing the Difference: Euler Interpolation vs. Quaternion SLERP
To prove why Dual Quaternions are the industry standard, we need to look at **Interpolation** (path planning). 

When a robot moves from Pose A to Pose B, the controller generates hundreds of intermediate micro-poses to ensure smooth motion. 
* If we simply draw a straight line between two sets of **Euler Angles** (Roll, Pitch, Yaw), the physical tool takes a warped, unpredictable path. The rotational velocity accelerates and decelerates wildly.
* If we use **Quaternion SLERP** (Spherical Linear Interpolation), the math calculates the shortest possible great-circle arc in 3D space. The tool moves at a perfectly constant velocity.

For a complex end-effector—especially something like a single-motor wrist with a separated tool axis—an unpredictable swing caused by Euler interpolation can easily result in the tool colliding with the workpiece or the robot's own chassis.

Run the cell below to race both algorithms. You will see the Euler path (Red) wobble and swing wide, while the Quaternion path (Green) takes the perfect, most direct arc between the two orientations.

[Another visualization tool for Euler vs Quaternion SLERP](https://arm-kinematics-webapp.vercel.app/#/interpolation)
[Raw Quaternion Visualization](https://arm-kinematics-webapp.vercel.app/#/quater)

In [30]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from scipy.spatial.transform import Slerp
from ipywidgets import interact, FloatSlider

# Define extreme Start and End Orientations (Roll, Pitch, Yaw)
# We push the Pitch near 90 degrees to show how badly Euler math degrades near Gimbal Lock
euler_start = [0, 0, 0]
euler_end = [120, 85, -120]

# Generate 100 steps for the trails
times = np.linspace(0, 1, 100)

# --- Pre-calculate the Euler Path ---
euler_interp_angles = np.linspace(euler_start, euler_end, 100)
rot_euler = R.from_euler('xyz', euler_interp_angles, degrees=True)
# Apply the rotations to a Z-axis tool vector
euler_trail = rot_euler.apply([0, 0, 1])

# --- Pre-calculate the Quaternion SLERP Path ---
rot_start = R.from_euler('xyz', euler_start, degrees=True)
rot_end = R.from_euler('xyz', euler_end, degrees=True)
key_rots = R.from_quat([rot_start.as_quat(), rot_end.as_quat()])
slerp = Slerp([0, 1], key_rots)

rot_slerp = slerp(times)
slerp_trail = rot_slerp.apply([0, 0, 1])

def plot_interpolation_race(t):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the full trails (Ghost lines)
    ax.plot(euler_trail[:, 0], euler_trail[:, 1], euler_trail[:, 2], color='red', alpha=0.3, linewidth=2, label='Euler Path')
    ax.plot(slerp_trail[:, 0], slerp_trail[:, 1], slerp_trail[:, 2], color='green', alpha=0.3, linewidth=2, label='SLERP Path')
    
    # Find the closest index for the current time t
    idx = int(t * 99)
    
    # Extract current vectors
    curr_euler = euler_trail[idx]
    curr_slerp = slerp_trail[idx]
    
    # Plot Start and End target vectors
    ax.quiver(0,0,0, *euler_trail[0], color='black', linewidth=1, alpha=0.5, linestyle='dashed')
    ax.quiver(0,0,0, *euler_trail[-1], color='black', linewidth=1, alpha=0.5, linestyle='dashed')
    
    # Plot the racing vectors
    ax.quiver(0,0,0, *curr_euler, color='red', linewidth=4)
    ax.quiver(0,0,0, *curr_slerp, color='green', linewidth=4)
    
    # Formatting
    ax.set_xlim([-1, 1]); ax.set_ylim([-1, 1]); ax.set_zlim([0, 1])
    ax.set_xlabel('X Axis'); ax.set_ylabel('Y Axis'); ax.set_zlabel('Z Axis')
    
    # Calculate the angular distance between the two vectors at this exact millisecond
    dot_product = np.dot(curr_euler, curr_slerp)
    divergence_deg = np.degrees(np.arccos(np.clip(dot_product, -1.0, 1.0)))
    
    ax.set_title(f"Time: {t*100:.0f}% | Current Vector Divergence: {divergence_deg:.1f}°")
    ax.legend(loc='upper left')
    
    # Set a good viewing angle to see the curve difference
    ax.view_init(elev=30, azim=45)
    plt.show()

# Setup UI Slider
interact(plot_interpolation_race,
         t=FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description='Time (t)'))

interactive(children=(FloatSlider(value=0.0, description='Time (t)', max=1.0, step=0.01), Output()), _dom_clas…

<function __main__.plot_interpolation_race(t)>

---

### Conclusion: The Foundation is Built
If you have made it this far, congratulations. You have just completed the most mathematically unforgiving aspect of robotics engineering. 

Look at the journey we just took:
* We started with basic **Forward Kinematics**, chaining 2D geometry to figure out where a tool is located in space.
* We built an **Analytical Solver**, proving how Kinematic Decoupling can solve our arm's custom architecture—even with a specialized single-motor wrist—perfectly, until it hits a singularity.
* We exposed the fatal flaws of Euler angles and the **Gimbal Lock** trap.
* We built a professional **Numerical Solver** using the Geometric Jacobian, integrating the Null Space for redundant control and Damped Least Squares to keep the system stable. We also ensured our kinematic logic was perfectly aligned by removing joint offsets so our controls zero out flawlessly.
* Finally, we proved why **Dual Quaternions** are the undisputed gold standard for modern robotic path planning.

You now possess the exact mathematical algorithms running inside the controllers of billion-dollar industrial robots and cutting-edge research manipulators. 

#### What's Next? Module 2: The Digital Twin
Up until this point, we have been plotting lines and arrows in a mathematical void. But to eventually train an intelligent AI, we cannot rely on Matplotlib. We need a physics engine. 

In **Module 2: The Digital Twin**, we will take the kinematic foundation we just built and inject it into a high-fidelity simulator. We will write the XML architecture to recreate our arm in **MuJoCo** (or Webots), spawning it into a virtual world complete with gravity, collision meshes, and simulated torque motors. 

#### The Horizon: Module 3: Reinforcement Learning
Once our digital twin is fully constructed and moving in the physics engine, we let the AI take over. In **Module 3**, we will wrap our simulation environment in a standard framework and use Reinforcement Learning to train an agent to control the arm purely through trial, error, and physics. 

The math is finished. It is time to build the simulation. See you in Module 2!